# 🌌 Stellar Classification — Exploratory Data Analysis

Classify each source as **GALAXY**, **QSO** (quasar), or **STAR** from photometric
and spectroscopic features in a *synthetic* SDSS-style dataset.

We use **Polars** for fast data manipulation and **Plotly** for interactive plots.

## Objectives
1. Profile the data: shapes, dtypes, missing values, duplicates.
2. Understand the (imbalanced) target distribution and pick a CV strategy.
3. Check **train/test consistency** (drift) — critical on a synthetic set.
4. Examine feature relationships, multicollinearity, and per-class separability.
5. Probe the synthetic generator's "gotchas" — and treat them as *signal*, not noise.
6. Engineer color indices and establish a **LightGBM baseline** with stratified CV.

In [ ]:
import polars as pl
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from pathlib import Path

pio.renderers.default = "iframe"
pio.templates.default = "plotly_white"  # consistent clean theme for all figures
SEED = 42  # one seed for all sampling/splits -> reproducible

# Resolve data dir for both local (../data) and Kaggle (/kaggle/input/<slug>) runs
_kaggle = Path("/kaggle/input")
_hits = sorted(_kaggle.rglob("train.csv")) if _kaggle.exists() else []
if _hits:
  DATA = _hits[0].parent
elif Path("../data/train.csv").exists():
  DATA = Path("../data")
else:
  raise FileNotFoundError(
      "Could not find train.csv. On Kaggle: click 'Add Input' and attach the "
      "competition dataset. Locally: place the CSVs in ../data/."
  )
print(f"Loading data from: {DATA}")
train = pl.read_csv(DATA / "train.csv")
test = pl.read_csv(DATA / "test.csv")
NUM_COLS = ["alpha", "delta", "u", "g", "r", "i", "z", "redshift"]
CAT_COLS = ["spectral_type", "galaxy_population"]
TARGET = "class"
print(f"train: {train.shape}   test: {test.shape}")

## 📊 1. Dataset Properties

Schema, missing values, duplicates, and a numeric summary before we trust anything.

In [2]:
# Schema (note: spectral_type / galaxy_population are categorical strings)
print("Schema:")
for name, dtype in train.schema.items():
    print(f"  {name:<18} {dtype}")

# Missing values — expect none in this synthetic set; confirm so we skip imputation
n_train_nulls = train.null_count().to_numpy().sum()
n_test_nulls = test.null_count().to_numpy().sum()
print(f"\nTotal nulls — train: {n_train_nulls}, test: {n_test_nulls}")

# Duplicate rows (excluding id) and duplicate ids
dup_rows = train.drop("id").is_duplicated().sum()
dup_ids = train["id"].is_duplicated().sum()
print(f"Duplicate feature rows: {dup_rows}   Duplicate ids: {dup_ids}")

# Numeric summary
train.select(NUM_COLS).describe()

Schema:
  id                 Int64
  alpha              Float64
  delta              Float64
  u                  Float64
  g                  Float64
  r                  Float64
  i                  Float64
  z                  Float64
  redshift           Float64
  spectral_type      String
  galaxy_population  String
  class              String

Total nulls — train: 0, test: 0
Duplicate feature rows: 0   Duplicate ids: 0


statistic,alpha,delta,u,g,r,i,z,redshift
str,f64,f64,f64,f64,f64,f64,f64,f64
"""count""",577347.0,577347.0,577347.0,577347.0,577347.0,577347.0,577347.0,577347.0
"""null_count""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""mean""",181.616673,21.834654,22.441926,21.007273,19.962811,19.378911,19.041136,0.723135
"""std""",96.242941,18.93357,2.018135,1.795426,1.648964,1.580059,1.584365,0.81007
"""min""",0.011684,-17.966988,-0.139225,13.535483,12.579407,11.962781,11.682803,-0.00997
"""25%""",132.16151,2.474226,20.97709,19.865006,18.820672,18.306851,17.973193,0.181053
"""50%""",188.681465,21.484412,22.570222,21.46782,20.431153,19.631642,19.188598,0.497525
"""75%""",231.829703,36.988337,23.869104,22.292719,21.164098,20.608198,20.162112,0.881393
"""max""",359.99981,79.158322,28.253263,27.620208,25.254499,27.910853,26.826867,7.01078


## 🎯 2. Target Distribution & CV Strategy

The classes are **imbalanced**, so:
- We must use **stratified** cross-validation to keep fold class ratios stable.
- The official metric is **balanced accuracy** (macro-averaged recall): each class
  is weighted equally, so the rarer STAR/QSO classes count as much as GALAXY and
  predicting the majority class is not rewarded. We track it as the primary CV
  metric, alongside macro-F1 and log-loss.

In [3]:
class_dist = (
    train.group_by(TARGET)
    .agg(pl.len().alias("count"))
    .with_columns((pl.col("count") / len(train) * 100).round(2).alias("percentage"))
    .sort("count", descending=True)
)
print(class_dist)

fig = px.bar(
    class_dist.to_pandas(),
    x=TARGET, y="count", color=TARGET, text="percentage",
    title="Target Class Distribution",
    color_discrete_sequence=px.colors.qualitative.Prism,
)
fig.update_traces(texttemplate="%{text}%", textposition="outside")
fig.show()

shape: (3, 3)
┌────────┬────────┬────────────┐
│ class  ┆ count  ┆ percentage │
│ ---    ┆ ---    ┆ ---        │
│ str    ┆ u32    ┆ f64        │
╞════════╪════════╪════════════╡
│ GALAXY ┆ 377480 ┆ 65.38      │
│ QSO    ┆ 117143 ┆ 20.29      │
│ STAR   ┆ 82724  ┆ 14.33      │
└────────┴────────┴────────────┘


## 🔍 3. Train/Test Consistency (Drift Check)

On a synthetic dataset the test set can be generated with shifted distributions.
We (a) compare categorical level proportions and numeric medians, and (b) run a quick
**adversarial validation**: train a classifier to tell train from test. An ROC-AUC
near **0.5** means the two sets are indistinguishable (good); much higher means drift
that could break CV-to-LB correlation.

In [4]:
# (a) Categorical level proportions: train vs test
for c in CAT_COLS:
    tr = train[c].value_counts(normalize=True).rename({"proportion": "train"})
    te = test[c].value_counts(normalize=True).rename({"proportion": "test"})
    merged = tr.join(te, on=c, how="full", coalesce=True).sort("train", descending=True)
    print(f"=== {c} (proportion) ===")
    print(merged.with_columns(pl.col(["train", "test"]).round(4)))
    print()

# (b) Numeric medians: train vs test
med = pl.concat([
    train.select(NUM_COLS).median().with_columns(pl.lit("train").alias("set")),
    test.select(NUM_COLS).median().with_columns(pl.lit("test").alias("set")),
])
print("=== Numeric medians ===")
print(med)

=== spectral_type (proportion) ===
shape: (4, 3)
┌───────────────┬────────┬────────┐
│ spectral_type ┆ train  ┆ test   │
│ ---           ┆ ---    ┆ ---    │
│ str           ┆ f64    ┆ f64    │
╞═══════════════╪════════╪════════╡
│ M             ┆ 0.5254 ┆ 0.5248 │
│ A/F           ┆ 0.2115 ┆ 0.2119 │
│ G/K           ┆ 0.188  ┆ 0.1891 │
│ O/B           ┆ 0.0751 ┆ 0.0742 │
└───────────────┴────────┴────────┘

=== galaxy_population (proportion) ===
shape: (2, 3)
┌───────────────────┬────────┬───────┐
│ galaxy_population ┆ train  ┆ test  │
│ ---               ┆ ---    ┆ ---   │
│ str               ┆ f64    ┆ f64   │
╞═══════════════════╪════════╪═══════╡
│ Red_Sequence      ┆ 0.5535 ┆ 0.552 │
│ Blue_Cloud        ┆ 0.4465 ┆ 0.448 │
└───────────────────┴────────┴───────┘

=== Numeric medians ===
shape: (2, 9)
┌────────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬──────────┬───────┐
│ alpha      ┆ delta     ┆ u         ┆ g         ┆ … ┆ i         ┆ z         ┆ redshift 

In [5]:
# Adversarial validation: can a model separate train from test?
from lightgbm import LGBMClassifier
from sklearn.model_selection import cross_val_score

av_cols = NUM_COLS + CAT_COLS
av = pl.concat([
    train.select(av_cols).with_columns(pl.lit(0).alias("is_test")),
    test.select(av_cols).with_columns(pl.lit(1).alias("is_test")),
]).sample(n=120_000, seed=SEED)  # subsample for speed

X_av = av.select(av_cols).to_pandas()
for c in CAT_COLS:
    X_av[c] = X_av[c].astype("category")
y_av = av["is_test"].to_numpy()

av_clf = LGBMClassifier(n_estimators=200, learning_rate=0.05, num_leaves=31,
                        random_state=SEED, verbose=-1)
auc = cross_val_score(av_clf, X_av, y_av, cv=3, scoring="roc_auc").mean()
print(f"Adversarial validation ROC-AUC: {auc:.4f}  (0.5 = no drift, 1.0 = total drift)")

Adversarial validation ROC-AUC: 0.5006  (0.5 = no drift, 1.0 = total drift)


## 🔗 4. Correlation & Multicollinearity

The five photometric bands `u, g, r, i, z` measure brightness in adjacent wavelength
windows, so they are strongly correlated. This motivates the **color indices** below.

In [6]:
corr = train.select(NUM_COLS).corr().to_numpy()  # numpy once -> clean indexing

fig = px.imshow(
    corr, x=NUM_COLS, y=NUM_COLS,
    color_continuous_scale="RdBu_r", zmin=-1, zmax=1,
    title="Feature Correlation Matrix",
)
for a in range(len(NUM_COLS)):
    for b in range(len(NUM_COLS)):
        fig.add_annotation(
            x=NUM_COLS[b], y=NUM_COLS[a], text=f"{corr[a, b]:.2f}", showarrow=False,
            font=dict(color="white" if abs(corr[a, b]) > 0.5 else "black"),
        )
fig.show()

## 🌠 5. Gotcha #1 — The Redshift Wrinkle

Physically, **stars** in our galaxy sit at redshift ≈ 0, while **galaxies** and
**quasars (QSO)** can have large redshifts. `redshift` should therefore be the single
most powerful feature. Note the dataset includes small **negative** redshifts — these
are physical (blueshift / noise near zero), *not* errors to clip.

In [7]:
redshift_stats = train.group_by(TARGET).agg(
    pl.col("redshift").min().alias("min"),
    pl.col("redshift").quantile(0.01).alias("p1"),
    pl.col("redshift").median().alias("median"),
    pl.col("redshift").quantile(0.99).alias("p99"),
    pl.col("redshift").max().alias("max"),
).sort("median")
print(redshift_stats)

fig = px.box(
    train.sample(n=50_000, seed=SEED).to_pandas(),
    x=TARGET, y="redshift", color=TARGET,
    title="Redshift Distribution by Class (50k sample)",
    color_discrete_sequence=px.colors.qualitative.Safe,
)
fig.show()

stars = train.filter(pl.col(TARGET) == "STAR")
high_z = stars.filter(pl.col("redshift") > 0.05)
print(f"Stars: {len(stars)}   with redshift > 0.05: {len(high_z)} "
      f"({len(high_z) / len(stars) * 100:.2f}%)")

shape: (3, 6)
┌────────┬───────────┬───────────┬──────────┬──────────┬──────────┐
│ class  ┆ min       ┆ p1        ┆ median   ┆ p99      ┆ max      │
│ ---    ┆ ---       ┆ ---       ┆ ---      ┆ ---      ┆ ---      │
│ str    ┆ f64       ┆ f64       ┆ f64      ┆ f64      ┆ f64      │
╞════════╪═══════════╪═══════════╪══════════╪══════════╪══════════╡
│ STAR   ┆ -0.00997  ┆ -0.008731 ┆ 0.056492 ┆ 0.237864 ┆ 5.445217 │
│ GALAXY ┆ -0.009934 ┆ 0.005185  ┆ 0.48196  ┆ 1.308074 ┆ 6.860273 │
│ QSO    ┆ 0.0001    ┆ 0.0001    ┆ 1.798886 ┆ 5.161866 ┆ 7.01078  │
└────────┴───────────┴───────────┴──────────┴──────────┴──────────┘


Stars: 82724   with redshift > 0.05: 45010 (54.41%)


## 🏷️ 6. Gotchas #2 & #3 — Categorical Artifacts as Signal

The generator assigns two categoricals that are physically odd:

- **`galaxy_population`** (`Red_Sequence`, `Blue_Cloud`) — a galaxy concept, yet it is
  also assigned to STARs and QSOs.
- **`spectral_type`** — here it is **binned** into `O/B`, `A/F`, `G/K`, `M` (not the raw
  O–M letters), and is applied to galaxies/QSOs too, which is unphysical.

For a classifier on synthetic data this is **not a reason to drop them** — what matters
is whether the value is distributed *differently across classes*. The normalized
crosstabs below (proportion within each class) reveal exploitable signal.

In [8]:
def class_crosstab(df, col):
    ct = df.group_by([TARGET, col]).agg(pl.len().alias("count"))
    ct = ct.with_columns(
        (pl.col("count") / pl.col("count").sum().over(TARGET) * 100).round(1).alias("pct_of_class")
    )
    # pivot to class x level grid of within-class percentages
    return ct.pivot(values="pct_of_class", index=col, on=TARGET).fill_null(0)

for c in CAT_COLS:
    print(f"=== {c}: % within each class ===")
    print(class_crosstab(train, c))
    print()

=== spectral_type: % within each class ===
shape: (4, 4)
┌───────────────┬────────┬──────┬──────┐
│ spectral_type ┆ GALAXY ┆ QSO  ┆ STAR │
│ ---           ┆ ---    ┆ ---  ┆ ---  │
│ str           ┆ f64    ┆ f64  ┆ f64  │
╞═══════════════╪════════╪══════╪══════╡
│ M             ┆ 76.3   ┆ 3.3  ┆ 13.8 │
│ A/F           ┆ 6.4    ┆ 52.5 ┆ 44.0 │
│ G/K           ┆ 16.3   ┆ 17.9 ┆ 31.4 │
│ O/B           ┆ 1.0    ┆ 26.3 ┆ 10.8 │
└───────────────┴────────┴──────┴──────┘

=== galaxy_population: % within each class ===
shape: (2, 4)
┌───────────────────┬────────┬──────┬──────┐
│ galaxy_population ┆ GALAXY ┆ STAR ┆ QSO  │
│ ---               ┆ ---    ┆ ---  ┆ ---  │
│ str               ┆ f64    ┆ f64  ┆ f64  │
╞═══════════════════╪════════╪══════╪══════╡
│ Red_Sequence      ┆ 76.4   ┆ 26.8 ┆ 7.6  │
│ Blue_Cloud        ┆ 23.6   ┆ 73.2 ┆ 92.4 │
└───────────────────┴────────┴──────┴──────┘



## 🌈 7. Color Indices & Per-Feature Separability

Standard astronomy practice: replace correlated absolute magnitudes with **color
indices** (differences between adjacent bands), which capture the spectral shape and
shed the absolute-brightness scale. We add them to **both** train and test, then rank
every feature by **mutual information** with the target to see what actually separates
the classes.

In [9]:
def add_colors(df):
    return df.with_columns([
        (pl.col("u") - pl.col("g")).alias("u_g"),
        (pl.col("g") - pl.col("r")).alias("g_r"),
        (pl.col("r") - pl.col("i")).alias("r_i"),
        (pl.col("i") - pl.col("z")).alias("i_z"),
    ])

train = add_colors(train)
test = add_colors(test)  # keep train/test feature sets identical
COLOR_COLS = ["u_g", "g_r", "r_i", "i_z"]

sample_df = train.sample(n=5_000, seed=SEED).to_pandas()
fig = px.scatter(
    sample_df, x="g_r", y="r_i", color=TARGET, opacity=0.6,
    title="Color-Color Diagram: (g - r) vs (r - i) [5k sample]",
    color_discrete_sequence=px.colors.qualitative.Vivid,
)
fig.show()

In [10]:
from sklearn.feature_selection import mutual_info_classif

feat_cols = NUM_COLS + COLOR_COLS + CAT_COLS
mi_sample = train.sample(n=80_000, seed=SEED)
X_mi = mi_sample.select(feat_cols).to_pandas()
# encode categoricals as integer codes for MI; flag them as discrete
for c in CAT_COLS:
    X_mi[c] = X_mi[c].astype("category").cat.codes
discrete = [col in CAT_COLS for col in feat_cols]
mi = mutual_info_classif(X_mi, mi_sample[TARGET].to_numpy(),
                         discrete_features=discrete, random_state=SEED)

mi_df = (pl.DataFrame({"feature": feat_cols, "mutual_info": mi})
         .sort("mutual_info", descending=True))
print(mi_df)
fig = px.bar(mi_df.to_pandas(), x="mutual_info", y="feature", orientation="h",
             title="Mutual Information with Target (80k sample)")
fig.update_yaxes(categoryorder="total ascending")
fig.show()

shape: (14, 2)
┌───────────────┬─────────────┐
│ feature       ┆ mutual_info │
│ ---           ┆ ---         │
│ str           ┆ f64         │
╞═══════════════╪═════════════╡
│ redshift      ┆ 0.513472    │
│ g_r           ┆ 0.319941    │
│ spectral_type ┆ 0.300818    │
│ r_i           ┆ 0.215775    │
│ z             ┆ 0.210993    │
│ …             ┆ …           │
│ i             ┆ 0.156396    │
│ alpha         ┆ 0.120593    │
│ delta         ┆ 0.113649    │
│ r             ┆ 0.097713    │
│ i_z           ┆ 0.074864    │
└───────────────┴─────────────┘


## 🚀 8. Baseline Model — LightGBM with Stratified CV

A first honest score to beat. LightGBM handles the categoricals natively (no one-hot),
the folds are **stratified** to respect class imbalance, and we report **balanced
accuracy** (the competition metric), macro-F1, and log-loss plus an out-of-fold
confusion matrix and feature importances.

In [ ]:
from lightgbm import LGBMClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                             log_loss, confusion_matrix)

feat_cols = NUM_COLS + COLOR_COLS + CAT_COLS
classes = sorted(train[TARGET].unique().to_list())
cls_to_int = {c: i for i, c in enumerate(classes)}

X = train.select(feat_cols).to_pandas()
for c in CAT_COLS:
    X[c] = X[c].astype("category")
y = train[TARGET].replace_strict(cls_to_int).to_numpy()

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
oof = np.zeros((len(y), len(classes)))
importances = np.zeros(len(feat_cols))
bals, accs, f1s = [], [], []

for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
    model = LGBMClassifier(
        n_estimators=600, learning_rate=0.05, num_leaves=63,
        subsample=0.8, colsample_bytree=0.8, random_state=SEED, verbose=-1,
    )
    model.fit(X.iloc[tr_idx], y[tr_idx],
              eval_set=[(X.iloc[va_idx], y[va_idx])],
              eval_metric="multi_logloss")
    proba = model.predict_proba(X.iloc[va_idx])
    oof[va_idx] = proba
    pred = proba.argmax(1)
    bals.append(balanced_accuracy_score(y[va_idx], pred))
    accs.append(accuracy_score(y[va_idx], pred))
    f1s.append(f1_score(y[va_idx], pred, average="macro"))
    importances += model.feature_importances_ / skf.n_splits
    print(f"fold {fold}: balanced_acc={bals[-1]:.4f}  acc={accs[-1]:.4f}  "
          f"macroF1={f1s[-1]:.4f}")

oof_pred = oof.argmax(1)
print(f"\nOOF balanced accuracy : {balanced_accuracy_score(y, oof_pred):.4f}  "
      f"(+/- {np.std(bals):.4f})   <- competition metric")
print(f"OOF accuracy          : {accuracy_score(y, oof_pred):.4f}")
print(f"OOF macro-F1          : {f1_score(y, oof_pred, average='macro'):.4f}")
print(f"OOF log-loss          : {log_loss(y, oof):.4f}")

In [12]:
cm = confusion_matrix(y, oof_pred, normalize="true")
fig = px.imshow(cm, x=classes, y=classes, text_auto=".2f",
                color_continuous_scale="Blues",
                labels=dict(x="Predicted", y="True", color="Recall"),
                title="OOF Confusion Matrix (row-normalized)")
fig.show()

imp_df = (pl.DataFrame({"feature": feat_cols, "importance": importances})
          .sort("importance", descending=True))
fig = px.bar(imp_df.to_pandas(), x="importance", y="feature", orientation="h",
             title="LightGBM Feature Importance (mean gain over folds)")
fig.update_yaxes(categoryorder="total ascending")
fig.show()
print(imp_df)

shape: (14, 2)
┌───────────────────┬────────────┐
│ feature           ┆ importance │
│ ---               ┆ ---        │
│ str               ┆ f64        │
╞═══════════════════╪════════════╡
│ alpha             ┆ 14636.2    │
│ delta             ┆ 14548.4    │
│ redshift          ┆ 11712.6    │
│ r_i               ┆ 9020.8     │
│ g_r               ┆ 8636.2     │
│ …                 ┆ …          │
│ g                 ┆ 7106.0     │
│ r                 ┆ 6366.2     │
│ i                 ┆ 6318.8     │
│ galaxy_population ┆ 339.0      │
│ spectral_type     ┆ 176.2      │
└───────────────────┴────────────┘


## 📝 9. Summary & Next Steps

**Findings → modeling implications**
- **No missing values / duplicates** → no imputation or dedup needed.
- **Imbalanced target** (GALAXY ≫ QSO > STAR) → keep stratified CV and optimize for
  **balanced accuracy** (the official metric), which weights all classes equally.
- **`redshift` dominates** separability (stars ≈ 0, QSO high) — confirmed by MI and
  feature importance.
- **Photometric bands are collinear** → color indices add shape information cheaply.
- **Categorical "artifacts" are predictive** because their per-class distributions
  differ — keep `spectral_type` and `galaxy_population` as native LightGBM categoricals.
- **Drift**: adversarial-validation AUC near 0.5 means CV should track the LB.

**Next steps**
1. Tune LightGBM (and try XGBoost/CatBoost) and consider class weights.
2. Engineer more colors / band ratios; test interaction with redshift.
3. Build the submission pipeline; tune class weights / decision thresholds to
   maximize balanced accuracy.